# Quick Clustering Workflow

This notebook demonstrates a quick clustering workflow used in detecting the different types of pockets in a nanoporouse material. It uses a collection of default parameters and may not work perfectly for all materials. But, it is a good starting point for tuning. The steps used in this workflow are:
- Compute the grid
- Compute the local maxima in the grid
- Compute region segmentation
- Apply periodic boundary conditions
- Detect connectivity and construct a pore graph
- Add some basic properties to the nodes of the pore graph
- Make figures with all the information

## Tips and Tricks #1
All the possible arguments to the function can be looked at using the `help` function.

```python
help(mgr.quick_cluster)
```


## Imports

In [10]:
# add path to mofography
import os,sys
path_to_mgr = os.path.abspath('C:/Users/Arun Gopalan/OneDrive - The University of Manchester/my_cloned_repos/mofography')
sys.path.append(path_to_mgr)
import mofography as mgr

from ase.io import read

path_to_cif = os.path.join(path_to_mgr, 'examples', 'TestCIFs', 'P1-IRMOF-1.cif')


## Read the CIF file

In [11]:
mat_atoms = read(path_to_cif)

# figures = mgr.quick_cluster(mat_atoms, return_these_vars=[figures])

c:\Users\Public\anaconda3\envs\jl_env\lib\site-packages\ase\io\cif.py:401: UserWarning:

crystal system 'triclinic' is not interpreted for space group Spacegroup(1, setting=1). This may result in wrong setting!



## Run the quick cluster

Let's run the quick cluster with default params. By default, it returns only the figures. We can see later how to get other outputs from the workflow.

In [12]:
figures = mgr.quick_cluster(mat_atoms)

C:\Users\Arun Gopalan\OneDrive - The University of Manchester\my_cloned_repos\mofography\mofography\compute.py:327: UserWarning:

Some maxima excluded as they fell inside the mask_thickness

c:\Users\Public\anaconda3\envs\jl_env\lib\site-packages\ase\atoms.py:967: VisibleDeprecationWarning:

Use get_global_number_of_atoms() instead



No node props provided, using default set: [position, scaled_position, element and atomic_number]
No edge properties provided. Use the default set: [bond_order, bond_type]
isomorphic aromatic cycles is None, trying to calculate


C:\Users\Arun Gopalan\OneDrive - The University of Manchester\my_cloned_repos\mofography\mofography\visualize.py:763: UserWarning:

Optimal rotation is not uniquely or poorly defined for the given sets of vectors.



No return objects specified, possible options to pass in list_return_strings are:
dict_keys(['ase_atoms', 'spacing', 'mask_thickness', 'h', 'compactness', 'zoom_ratio', 'wall_thickness', 'isoval', 'bins', 'block_size', 'a', 'b', 'c', 'density', 'distance_threshold', 'minimum_overlap_fraction', 'make_figures', 'renderer', 'use_energy_for_clustering', 'sample_fraction', 'cscale', 'return_these_vars', 'np', 'inspect', 'covalent_radii', 'ase', 'da', 'zoom', 'regionprops', 'go', 'px', 'update_with_covalent_bonds', 'dgrid_from_atoms', 'regions_from_dgrid', 'apply_pbc', 'get_rag_from_regions_and_grid', 'add_pbc_to_rag', 'add_psd_to_rag', 'add_psd_to_rag_with_pbc', 'add_clustering_to_rag_pca', 'clustering_grid_from_regions', 'add_clustering_to_rag_energy', 'add_vdw_hist_to_rag', 'mix_lorentz_berthelot', 'plot_surface_dgrid', 'plot_framework_as_mesh', 'plot_cell', 'plot_maxima', 'plot_psds', 'white_layout', 'plot_vdw_hists', 'mat_updated', 'radii', 'grid', 'regions', 'maxima', 'maxima_radii', '

## The first figure

The first figure we are going to look at is quite loaded with information. It shows three isosurfaces (one on top of the other, so you have to toggle the top layers to see the bottom ones), the local maxima, the unit cell and the framework (atoms, bonds and aromatic rings). The three isosurfaces correspond to
- The raw region segmentation
- The region segmentation after applying periodic boundary conditions, where regions that are connected across the periodic boundary are merged to the same index
- The types of pockets, where all the pockets detected to be of the same type and shown in the same color. 

There is more information available in the figure if you hover over any of the surfaces or the framework atoms.

In [13]:
figures[0].show(renderer='browser')

Any of the plot traces can be recoverd using the data variable of the Plotly figure object. 
`fig.data`. For example, to just see the names of the traces use. Please refer to Plotly documentation for more informaton on the figure structures  https://plotly.com/python/figure-structure/

In [14]:
[d['name'] for d in figures[0].data]

['Raw segmentation',
 'PBC groups',
 'Cavity types',
 'Framework',
 'Aromatic rings',
 'Aromatic highlights',
 'Aromatic highlights',
 'Aromatic highlights',
 'Aromatic highlights',
 'Aromatic highlights',
 'Aromatic highlights',
 'Aromatic highlights',
 'Aromatic highlights',
 'Aromatic highlights',
 'Aromatic highlights',
 'Aromatic highlights',
 'Aromatic highlights',
 'cell',
 'local maxima']

To set a different color scheme, say for the `Cavity types` mesh trace to `Pastel`, you can do that also post generation.

In [15]:
fig = figures[0]
import plotly.express as px 
fig.data[2]['colorscale'] = px.colors.qualitative.Pastel

fig.show(renderer='browser')

## The second figure
 
The second figure returned simply contains the Cavity features that are used in clustering the pockets. One trace per PBC group. The traces are then colored by the cluster to which they belong to (shows on hover). This is quite useful for debugging and understanding the efficiency of our features in conjunction with the clustering algorithm.

In [16]:
figures[1].show(renderer='notebook_connected')

## Recovering additional outputs
We can grab additional variables with these names:
- grid: the dask grid of distances. To make it into a numpy array, use `grid.compute()`
- maxima: indices of the local maxima in the grid
- rag: region adjacency graph or the pore graph, which is a networkx graph


Let's exract the pore graph (region adjacency graph) and add that also to the first figure.

In [17]:
rag, figures = mgr.quick_cluster(mat_atoms, return_these_vars=['rag'])

C:\Users\Arun Gopalan\OneDrive - The University of Manchester\my_cloned_repos\mofography\mofography\compute.py:327: UserWarning:

Some maxima excluded as they fell inside the mask_thickness

c:\Users\Public\anaconda3\envs\jl_env\lib\site-packages\ase\atoms.py:967: VisibleDeprecationWarning:

Use get_global_number_of_atoms() instead



No node props provided, using default set: [position, scaled_position, element and atomic_number]
No edge properties provided. Use the default set: [bond_order, bond_type]
isomorphic aromatic cycles is None, trying to calculate


C:\Users\Arun Gopalan\OneDrive - The University of Manchester\my_cloned_repos\mofography\mofography\visualize.py:763: UserWarning:

Optimal rotation is not uniquely or poorly defined for the given sets of vectors.



Some return objects specified, possible options to pass in list_return_strings are:


In [18]:
rag_trace = mgr.plot_rag(rag, name='p-graph')
fig.add_trace(rag_trace)
fig.show(renderer='browser')

## Where to next

Now that you've seen the utility mofography by using it somewhat like a 'black box', you can start to dig deep into the key steps of this core workflow by going through the Basic Tutorials. You may also look at the Advanced Tutorials, if all you need is to apply mofography to a specific real-world problem similar to one detailed there.